In [3]:
import networkx as nx
from itertools import combinations
import pandas as pd

In [23]:
cdr = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/combined_real_cdr_for_cider.csv'
)
subscribers = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/combined_real_subscribers.csv'
)

/var/folders/y5/65jnz7bx5t3_rcnkb0xz6fvw0000gn/T/ipykernel_49062/1779454872.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  cdr = pd.read_csv(


In [24]:

contacts = pd.concat([
    cdr[['caller_msisdn', 'recipient_msisdn']].rename(columns={'caller_msisdn': 'ego', 'recipient_msisdn': 'alter'}),
    cdr[['recipient_msisdn', 'caller_msisdn']].rename(columns={'recipient_msisdn': 'ego', 'caller_msisdn': 'alter'}),
]).drop_duplicates()

alter_to_egos = contacts.dropna(how='any').groupby('alter')['ego'].apply(list)

shared_edges = set()
for egos in alter_to_egos:
    for u, v in combinations(egos, 2):
        shared_edges.add((min(u, v), max(u, v)))

# Build graph
G = nx.Graph()
G.add_nodes_from(contacts['ego'].unique())
G.add_edges_from(shared_edges)

sub_ids = set(subscribers['subscriber_id'])
G_sub = G.subgraph([n for n in G.nodes if n in sub_ids])

print(f"Nodes : {G_sub.number_of_nodes():,}")
print(f"Edges : {G_sub.number_of_edges():,}")
print(f"Density: {nx.density(G_sub):.4f}")

Nodes : 3,924
Edges : 120,516
Density: 0.0157


In [25]:
total = G_sub.number_of_nodes()
for k in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20]:
    count = sum(1 for n in G_sub.nodes if G_sub.degree(n) >= k)
    print(f"degree >= {k:>3}: {count:,} / {total:,} ({100*count/total:.1f}%)")

degree >=   1: 2,531 / 3,924 (64.5%)
degree >=   2: 1,875 / 3,924 (47.8%)
degree >=   3: 1,524 / 3,924 (38.8%)
degree >=   4: 1,288 / 3,924 (32.8%)
degree >=   5: 1,150 / 3,924 (29.3%)
degree >=   6: 1,062 / 3,924 (27.1%)
degree >=   7: 1,004 / 3,924 (25.6%)
degree >=   8: 959 / 3,924 (24.4%)
degree >=   9: 930 / 3,924 (23.7%)
degree >=  10: 908 / 3,924 (23.1%)
degree >=  20: 809 / 3,924 (20.6%)
